# 02 — Preprocessing & Feature Engineering
### AI-Powered Resume Screening & Candidate Role Recommendation
---
**Run 01_EDA.ipynb first** — this notebook loads `data/processed_resume.csv` produced there.
It outputs a train/test split and a fitted `preprocessor` Pipeline saved to `models/`.


In [1]:
import os, ast, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Libraries loaded.")


Libraries loaded.


## 1. Load Processed Data

In [2]:
resume_df = pd.read_csv('data/processed_resume.csv')

# Re-parse extracted_skills (CSV serialization turns lists to strings)
resume_df['extracted_skills'] = resume_df['extracted_skills'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

print("Resume shape:", resume_df.shape)
print("Columns:", list(resume_df.columns))


Resume shape: (10000, 14)
Columns: ['Resume ID', 'Resume Text', 'Education', 'Experience Years', 'Skills', 'Job Role', 'Category', 'clean_text', 'extracted_skills', 'skill_count', 'category_encoded', 'job_role_encoded', 'education_level', 'known_skill_ratio']


## 2. Feature Engineering Decisions

| Feature | Type | Reason |
|---|---|---|
| `clean_text` | TF-IDF (max_features=3000) | Rich text signal; 3000 captures enough vocabulary without being too sparse |
| `Experience Years` | StandardScaler | Continuous numeric — scale needed |
| `skill_count` | StandardScaler | Continuous numeric — scale needed |
| `education_level` | StandardScaler | Ordinal encoded upstream |

**Why 3000 TF-IDF features?** Resumes are text-heavy documents. 500 (the original default) was too conservative for 42 classes — it misses many distinguishing domain terms. 3000 provides 6× more vocabulary coverage while staying memory-efficient.

## 3. Train / Test Split (BEFORE Vectorizer Fit)

In [3]:
y_category = resume_df['category_encoded'].values

train_df, test_df, y_train, y_test = train_test_split(
    resume_df, y_category, test_size=0.2,
    random_state=RANDOM_STATE, stratify=y_category
)

print(f"Train : {train_df.shape[0]} rows")
print(f"Test  : {test_df.shape[0]} rows")
print(f"Classes: {resume_df['Category'].nunique()}")

# Save split indices for reproducibility
train_df.index.to_series().to_csv('data/train_indices.csv', index=False)
test_df.index.to_series().to_csv('data/test_indices.csv', index=False)
print("Split indices saved.")


Train : 8000 rows
Test  : 2000 rows
Classes: 42
Split indices saved.


## 4. Leak-Free Preprocessing Pipeline
`ColumnTransformer` wraps TF-IDF + StandardScaler. When this `preprocessor` is used inside a `Pipeline` with each classifier, sklearn **refits it per CV fold** — the vectorizer never sees test data.

In [4]:
text_col    = 'clean_text'
numeric_cols = ['Experience Years', 'skill_count', 'education_level']

preprocessor = ColumnTransformer(
    transformers=[
        ('tfidf', TfidfVectorizer(max_features=3000), text_col),
        ('num',   StandardScaler(),                  numeric_cols)
    ]
)

# Fit once on training data only — this fitted instance is for inspection/clustering
preprocessor.fit(train_df)
X_train_shape = preprocessor.transform(train_df).shape
X_test_shape  = preprocessor.transform(test_df).shape

print(f"Train feature matrix : {X_train_shape}")
print(f"Test feature matrix  : {X_test_shape}")


Train feature matrix : (8000, 768)
Test feature matrix  : (2000, 768)


## 5. Save Artifacts for Model Training Notebook

In [5]:
joblib.dump(preprocessor, 'models/preprocessor.pkl')
train_df.to_csv('data/train_df.csv', index=False)
test_df.to_csv('data/test_df.csv', index=False)

import numpy as np
np.save('data/y_train.npy', y_train)
np.save('data/y_test.npy',  y_test)

print("Saved: models/preprocessor.pkl")
print("Saved: data/train_df.csv, data/test_df.csv")
print("Saved: data/y_train.npy, data/y_test.npy")
print("\n✅ 02_Preprocessing_Feature_Engineering.ipynb complete — run 03_Model_Training.ipynb next.")


Saved: models/preprocessor.pkl
Saved: data/train_df.csv, data/test_df.csv
Saved: data/y_train.npy, data/y_test.npy

✅ 02_Preprocessing_Feature_Engineering.ipynb complete — run 03_Model_Training.ipynb next.
